# Notebook 02 — Element Extraction (Revised: + Multi-Word Chunks)

**Project:** Scènes à Faire Computational Framework (v4 rebuild)  
**Purpose:** Extract narrative elements from the union corpus using spaCy. This revision adds **multi-word noun chunk extraction** alongside the existing single-word lemma extraction.

**Changes from v1 of this notebook:**

The Sonnet 4.6 validation in Notebook 02b revealed that the biggest extraction gap was multi-word narrative phrases: 'haunted house,' 'final girl,' 'femme fatale,' 'mad scientist,' 'love triangle,' 'car chase,' 'alien planet,' 'mentor figure.' These are exactly the kinds of stock elements scènes à faire doctrine emphasizes (Restatement §18: 'standard, stock, customary, stereotypical, or common' settings, characters, and situations are usually multi-word). The previous extraction collapsed them to their root noun ('haunted house' → 'house'), losing the modifier that carried the genre signal.

**This notebook extracts four element categories:**
1. `noun` — single-word root lemmas (as before): `house`, `sheriff`, `zombie`
2. `chunk` — multi-word noun phrases (NEW): `haunted house`, `corrupt sheriff`, `final girl`
3. `verb` — single-word verb lemmas (as before): `kill`, `marry`, `investigate`
4. `entity_*` — typed named entities (as before): `entity_GPE: tokyo`, `entity_ORG: cia`

Single-word and multi-word forms compete independently in the stockiness rankings. 'House' and 'haunted house' get separate scores. The chunk version typically wins for Horror because it's more distinctive.

**Multi-word chunk extraction rules:**
- Drop determiners (`the`, `a`, `this`), pronouns, possessives (`his`, `her`), punctuation
- Drop stopwords from the chunk content
- Keep adjective + noun, compound noun, and short premodifier sequences
- Lemmatize each retained token (plural → singular)
- Only emit if 2-4 content tokens remain (1-token chunks already captured as `noun`; 5+ usually too specific)
- Same PERSON entity exclusion as single-word extraction

**Runtime:** the additional chunk extraction adds ~10-20% to spaCy processing time. Total expected runtime: 75-130 minutes on default CPU. Existing chunk cache from v1 is INCOMPATIBLE with the new extraction — we clear it and re-run.

**Outputs (same paths as v1, overwrites):**
- `elements_by_film.parquet` — one row per film with separate list columns for nouns, chunks, verbs, entities
- `element_index.parquet` — flattened, one row per (film, element, type) triple

## 1. Setup

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install -q spacy pyarrow inflect
!python -m spacy download en_core_web_lg

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl (400.7 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [11]:
import os
import re
import html
import pickle
import shutil
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import spacy
import inflect

In [12]:
PROJECT_ROOT = '/content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis'

UNION_PATH = f'{PROJECT_ROOT}/v4_rebuild/01c_union/movies_union.parquet'

OUTPUT_DIR = f'{PROJECT_ROOT}/v4_rebuild/02_elements'
CACHE_DIR  = f'{OUTPUT_DIR}/cache'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# CLEAR the v1 cache. The new extraction schema is different, so old chunks are invalid.
if os.path.exists(CACHE_DIR):
    print(f'Clearing v1 cache at {CACHE_DIR}...')
    shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir ready: {CACHE_DIR}')

assert os.path.exists(UNION_PATH), 'Union parquet missing. Run Notebook 01c first.'

Clearing v1 cache at /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02_elements/cache...
Cache dir ready: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02_elements/cache


## 2. Load the Union Corpus

In [13]:
union = pd.read_parquet(UNION_PATH)
print(f'Loaded {len(union):,} films')
print(f'Columns: {list(union.columns)}')

Loaded 45,958 films
Columns: ['union_id', 'name', 'year', 'decade', 'plot', 'char_count', 'word_count', 'token_count', 'genres_canonical', 'n_canonical_genres', 'corpus_source', 'cmu_wiki_id', 'kaggle_wiki_id']


## 3. Clean Plot Text (same regex as v1)

In [14]:
def clean_plot(text):
    """Strip MediaWiki markup, citations, HTML entities, and template residue."""
    if not isinstance(text, str) or not text:
        return ''
    # Strip {{templates}} (nested, apply 3 passes)
    for _ in range(3):
        text = re.sub(r'\{\{[^{}]*\}\}', '', text)
    # Strip <ref>...</ref> and self-closing <ref/>
    text = re.sub(r'<ref[^>]*>.*?</ref>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<ref[^/]*/>', '', text, flags=re.IGNORECASE)
    # Wikipedia links: [[target|display]] -> display; [[target]] -> target
    text = re.sub(r'\[\[([^\[\]|]+)\|([^\[\]]+)\]\]', r'\2', text)
    text = re.sub(r'\[\[([^\[\]]+)\]\]', r'\1', text)
    # Orphaned brackets (the zomby/Jeet bug)
    text = re.sub(r'\[\[+', '', text)
    text = re.sub(r'\]\]+', '', text)
    # Citation markers
    text = re.sub(r'\[citation needed\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\d+\]', '', text)
    # HTML entities - html.unescape covers all named and numeric entities
    # (&mdash;, &ndash;, &hellip;, &#8217;, ...), not a hand-picked few
    text = html.unescape(text)
    # Any remaining HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Cleaning plot text for all 45,958 films...')
union['plot_clean'] = union['plot'].apply(clean_plot)

before_drop = len(union)
union = union[union['plot_clean'].str.len() > 0].reset_index(drop=True)
print(f'Films retained after cleaning: {len(union):,} (dropped {before_drop - len(union)} empty)')

Cleaning plot text for all 45,958 films...
Films retained after cleaning: 45,958 (dropped 0 empty)


## 4. spaCy Setup with Stopwords and Filters

In [15]:
NOISE_NOUNS = {
    'thing', 'something', 'anything', 'nothing', 'everything',
    'someone', 'anyone', 'everyone', 'one', 'ones',
    'way', 'ways', 'sort', 'sorts', 'kind', 'kinds', 'type', 'types',
    'fact', 'facts', 'matter', 'matters', 'case', 'cases',
    'lot', 'lots', 'bit', 'bits', 'part', 'parts',
    'end', 'ends', 'beginning', 'middle', 'start',
    'number', 'numbers', 'amount', 'amounts',
    'moment', 'moments', 'point', 'points', 'while',
    'story', 'stories', 'tale', 'tales', 'plot',
    'scene', 'scenes', 'film', 'films', 'movie', 'movies',
    'character', 'characters',
    'person', 'people', 'guy', 'guys', 'others', 'other',
}

NOISE_VERBS = {
    'be', 'have', 'do', 'will', 'would', 'shall', 'should',
    'can', 'could', 'may', 'might', 'must',
    'get', 'got', 'make', 'made', 'take', 'took', 'give', 'gave',
    'put', 'go', 'went', 'come', 'came', 'let', 'leave', 'left',
    'bring', 'brought', 'keep', 'kept', 'hold', 'held',
    'say', 'said', 'tell', 'told', 'ask', 'asked', 'answer',
    'speak', 'spoke', 'talk', 'called', 'call',
    'see', 'saw', 'look', 'watch', 'notice', 'observe',
    'hear', 'heard', 'listen',
    'know', 'knew', 'think', 'thought', 'believe', 'believed',
    'feel', 'felt', 'wonder', 'wondered', 'consider',
    'want', 'wanted', 'need', 'needed', 'wish', 'wished',
    'try', 'tried', 'seem', 'seemed', 'appear', 'appeared',
    'help', 'helped', 'use', 'used', 'work', 'worked',
    'turn', 'turned', 'become', 'became', 'begin', 'began',
    'continue', 'continued', 'decide', 'decided',
    'show', 'shown', 'reveal', 'revealed', 'explain', 'explained',
    'meet', 'met', 'find', 'found',
}

USEFUL_ENTITY_TYPES = {
    'GPE', 'LOC', 'FAC', 'ORG', 'PRODUCT',
    'EVENT', 'WORK_OF_ART', 'NORP'
}

# Chunk extraction constraints
MIN_CHUNK_TOKENS = 2  # multi-word only (single words already captured as nouns)
MAX_CHUNK_TOKENS = 4  # cap at 4 content words to avoid over-specific phrases

print('Loading en_core_web_lg...')
nlp = spacy.load('en_core_web_lg')
print(f'Loaded. Pipeline: {nlp.pipe_names}')

# ----------------------------------------------------------------------
# Canonicalization helpers for entity and noun lemmatization
# ----------------------------------------------------------------------
# spaCy's default behavior handles nouns and verbs via its lemmatizer
# (so 'kills' -> 'kill', 'running' -> 'run'). But it has two consistency
# gaps that fragment our extraction:
#
#   1. Named-entity recognition preserves the literal surface form of
#      the span. For nationality entities ('the Germans', 'the
#      Americans', 'the Indians') this produces both 'germans' and
#      'german' as distinct entity_NORP rows in element_index, splitting
#      the framework's per-cell counts across morphological variants.
#
#   2. spaCy sometimes tags common-noun plurals ('Germans', 'Mexicans')
#      as PROPN (proper noun), in which case lemma_ returns the surface
#      form unchanged rather than the singular. This produces 'germans'
#      and 'german' as distinct noun rows.
#
# We apply inflect.singular_noun() as a canonicalization layer:
#
#   - For NORP entities: singularize each token in the entity span
#     before joining ('germans' -> 'german', 'palestinians' -> 'palestinian').
#   - For nouns (including chunk head nouns): apply singular_noun as a
#     safety net after spaCy's lemmatizer to catch the PROPN
#     inconsistency case.
#
# Other entity types (GPE, LOC, FAC, ORG, PRODUCT, EVENT, WORK_OF_ART)
# are preserved literally because lemmatizing them would corrupt them
# ('United States' -> 'United State' is wrong).
_INFLECT = inflect.engine()
LEMMATIZE_ENTITY_TYPES = {'NORP'}


@lru_cache(maxsize=20000)
def try_singularize(word):
    """Attempt to singularize an English plural noun.

    Returns the singularized form or None if inflect cannot process it.
    CRITICAL: only call this on tokens that spaCy has tagged as
    Number=Plur. inflect.singular_noun() aggressively strips trailing
    's' even from singular words ending in 's' or 'ss' ('boss' -> 'bos',
    'press' -> 'pres', 'glass' -> 'glas', 'address' -> 'addres'). We
    rely on spaCy's morphological analysis to gate the call so we only
    transform actual plurals.
    """
    result = _INFLECT.singular_noun(word)
    return result if result else None


def is_plural(spacy_token):
    """True iff spaCy's morphological analysis tagged this token as plural."""
    return 'Plur' in spacy_token.morph.get('Number')


Loading en_core_web_lg...
Loaded. Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 5. Element Extraction Function (Now with Multi-Word Chunks)

Given a spaCy Doc, return four element lists. The new piece is the chunk extraction: for each noun chunk, build a cleaned multi-word phrase by keeping content tokens (lemmatized) and dropping function words. Emit only if 2-4 content tokens remain.

In [16]:
# Token POS tags to skip when building multi-word chunks
FUNCTION_POS = {'DET', 'PRON', 'PART', 'PUNCT', 'SPACE', 'AUX', 'CCONJ', 'SCONJ', 'ADP'}

def build_chunk_phrase(chunk, person_token_idxs):
    """Clean a spaCy noun_chunk into a multi-word phrase.

    Returns the cleaned lowercase phrase if 2-4 content tokens remain, else None.
    Only the head noun is lemmatized; modifier tokens keep their surface form
    so adjectival participles ("abandoned", "haunted") are not verb-stemmed.
    """
    root_i = chunk.root.i
    content_tokens = []
    for tok in chunk:
        # Skip if this token is part of a PERSON entity
        if tok.i in person_token_idxs:
            return None  # bail entirely if chunk contains a character name
        # Skip function words
        if tok.pos_ in FUNCTION_POS:
            continue
        # Skip stopwords
        if tok.is_stop:
            continue
        # Skip non-alphabetic tokens
        if not tok.text.isalpha():
            continue
        # Head noun: lemmatize for plural-to-singular normalization.
        # Modifiers: keep the lowercased surface form. Lemmatizing modifiers
        # corrupts adjectival past participles - spaCy inconsistently tags
        # "abandoned"/"haunted"/"wanted" as VBN, whose lemma is the bare verb,
        # which mangles the phrase and fragments its count across two forms.
        if tok.i == root_i:
            form = tok.lemma_.lower().strip()
            # Same triple-gate PROPN safety net as the noun extraction
            if (form and form.endswith('s') and is_plural(tok)
                    and form == tok.text.lower()):
                singularized = try_singularize(form)
                if singularized:
                    form = singularized
        else:
            form = tok.text.lower().strip()
        # Skip very short tokens
        if len(form) < 2:
            continue
        content_tokens.append(form)

    n = len(content_tokens)
    if n < MIN_CHUNK_TOKENS or n > MAX_CHUNK_TOKENS:
        return None

    phrase = ' '.join(content_tokens)
    # Final noise check: the phrase as a whole isn't in the noise set
    # (we don't filter against NOISE_NOUNS at the multi-word level because
    # 'haunted house' contains 'house' which is fine as a chunk)
    return phrase

def extract_elements(doc):
    """Extract four categories of narrative elements from a spaCy Doc.
    Returns: (nouns, chunks, verbs, entities)
    """
    nouns = []
    chunks = []
    verbs = []
    entities = []

    # PERSON entity token indices for exclusion
    person_token_idxs = set()
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            for tok in ent:
                person_token_idxs.add(tok.i)

    # Iterate noun chunks once for BOTH the root-lemma and the multi-word extraction
    for chunk in doc.noun_chunks:
        root = chunk.root

        # --- Single-word noun ---
        if root.i not in person_token_idxs and root.pos_ not in ('PRON', 'DET') and not root.is_stop:
            lemma = root.lemma_.lower().strip()
            # PROPN safety net: spaCy sometimes tags common-noun plurals
            # ('Germans', 'Mexicans') as PROPN, in which case lemma_
            # returns the unchanged plural surface form. We singularize
            # only when ALL THREE conditions hold:
            #   (i)   the lemma ends in 's'
            #   (ii)  spaCy's morph says the token is plural
            #   (iii) spaCy did NOT already lemmatize (lemma == surface)
            # Without (iii), spaCy-lemmatized words like 'dresses' -> 'dress'
            # would be re-singularized by inflect to 'dres'. Without (ii),
            # 'boss' / 'glass' / 'press' would be stripped. The combined
            # gate catches exactly the PROPN-inconsistency case.
            if (lemma and lemma.endswith('s') and is_plural(root)
                    and lemma == root.text.lower()):
                singularized = try_singularize(lemma)
                if singularized:
                    lemma = singularized
            if lemma and len(lemma) >= 3 and lemma.isalpha() and lemma not in NOISE_NOUNS:
                nouns.append(lemma)

        # --- Multi-word chunk (NEW) ---
        phrase = build_chunk_phrase(chunk, person_token_idxs)
        if phrase is not None:
            chunks.append(phrase)

    # Verbs from main verb tokens (unchanged)
    for token in doc:
        if token.pos_ != 'VERB' or token.is_stop:
            continue
        lemma = token.lemma_.lower().strip()
        if not lemma or len(lemma) < 3 or not lemma.isalpha():
            continue
        if lemma in NOISE_VERBS:
            continue
        verbs.append(lemma)

    # Named entities
    for ent in doc.ents:
        if ent.label_ not in USEFUL_ENTITY_TYPES:
            continue
        entity_text = ent.text.lower().strip()
        # Strip leading articles spaCy sometimes includes in the NER span
        # ("a Rolls-Royce" -> "rolls-royce") and stray quote characters
        entity_text = re.sub(r'^(the|a|an)\s+', '', entity_text)
        entity_text = entity_text.strip('"\'').strip()
        if len(entity_text) < 3:
            continue
        # Canonicalize NORP entities by singularizing plural tokens.
        # Without this, 'germans' and 'german' both appear as distinct
        # entity_NORP elements and the framework treats them as separate
        # conventions (germans = 725 instances, german = 2,237 instances,
        # four-row split with the noun variants). After canonicalization
        # both fold into 'german'. We re-iterate over the spaCy tokens
        # in the entity span (rather than splitting entity_text on
        # whitespace) so we can use each token's morphological analysis
        # to gate the singularization, avoiding the inflect bug where
        # singular words ending in 's' get their 's' stripped.
        if ent.label_ in LEMMATIZE_ENTITY_TYPES:
            parts = []
            for tok in ent:
                # Skip the leading articles already stripped above
                if tok.text.lower() in {'a', 'an', 'the'}:
                    continue
                tok_text = tok.text.lower().strip('"\'').strip()
                if tok_text and tok_text.endswith('s') and is_plural(tok):
                    singularized = try_singularize(tok_text)
                    if singularized:
                        tok_text = singularized
                if tok_text:
                    parts.append(tok_text)
            entity_text = ' '.join(parts).strip()
            if len(entity_text) < 3:
                continue
        entities.append({'text': entity_text, 'label': ent.label_})

    return nouns, chunks, verbs, entities

In [17]:
# Quick smoke test on a few real plots to inspect chunk output
sample_indices = [0, 100, 500, 1000]
for idx in sample_indices:
    if idx >= len(union):
        continue
    row = union.iloc[idx]
    doc = nlp(row['plot_clean'][:3000])
    nouns, chunks, verbs, entities = extract_elements(doc)
    print(f'\n--- {row["name"]} ({row["primary_genre"] if "primary_genre" in row else row.get("genres_canonical", ["?"])[0] if isinstance(row.get("genres_canonical"), list) and row.get("genres_canonical") else "?"}) ---')
    print(f'  Nouns (first 10):   {nouns[:10]}')
    print(f'  Chunks (first 10):  {chunks[:10]}')
    print(f'  Verbs (first 10):   {verbs[:10]}')


--- The Hunger Games (?) ---
  Nouns (first 10):   ['nation', 'panem', 'capitol', 'district', 'punishment', 'rebellion', 'district', 'boy', 'girl', 'age']
  Chunks (first 10):  ['wealthy capitol', 'poorer district', 'past rebellion', 'annual hunger game', 'sole survivor', 'older sister', 'district tribute', 'frequently drunk mentor', 'career tribute', 'special academy']
  Verbs (first 10):   ['consist', 'provide', 'select', 'fight', 'reward', 'choose', 'starve', 'accompany', 'warn', 'train']

--- Three Little Bops (?) ---
  Nouns (first 10):   ['cartoon', 'display', 'book', 'pig', 'pipe', 'jig', 'day', 'pig', 'instrument', 'bops']
  Chunks (first 10):  ['little pig', 'dance jig', 'present day', 'modern instrument', 'little bops', 'big bad wolf', 'choice instrument', 'straw house', 'dew drop inn', 'dew drop inn']
  Verbs (first 10):   ['open', 'play', 'focus', 'play', 'perform', 'prove', 'state', 'join', 'happen', 'play']

--- Picknick mit Weissmann (?) ---
  Nouns (first 10):   ['phon

## 6. Batch Extraction with Checkpointing

Process the corpus in chunks of 2,000 films. Each completed chunk is pickled to `cache/chunk_NNNN.pkl`. If Colab disconnects, the loop restarts and skips cached chunks.

**Estimated runtime:** 75-130 minutes for 45,958 films on default CPU. The chunk extraction adds maybe 10-20% over the original noun-only extraction.

In [18]:
CHUNK_SIZE = 2000
SPACY_BATCH_SIZE = 100
total_films = len(union)
n_chunks = (total_films + CHUNK_SIZE - 1) // CHUNK_SIZE

completed_chunks = set()
for f in os.listdir(CACHE_DIR):
    m = re.match(r'chunk_(\d+)\.pkl', f)
    if m:
        completed_chunks.add(int(m.group(1)))

print(f'Total films:       {total_films:,}')
print(f'Total chunks:      {n_chunks}')
print(f'Cached chunks:     {len(completed_chunks)}')
print(f'To process:        {n_chunks - len(completed_chunks)}')

Total films:       45,958
Total chunks:      23
Cached chunks:     0
To process:        23


In [19]:
for chunk_idx in range(n_chunks):
    if chunk_idx in completed_chunks:
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_films)
    chunk_df = union.iloc[start:end]

    print(f'\nProcessing chunk {chunk_idx+1}/{n_chunks} (films {start}–{end-1})')

    texts = chunk_df['plot_clean'].tolist()
    union_ids = chunk_df['union_id'].tolist()

    results = []
    docs = nlp.pipe(texts, batch_size=SPACY_BATCH_SIZE)

    for i, doc in enumerate(tqdm(docs, total=len(texts), desc=f'chunk {chunk_idx}')):
        try:
            nouns, chunks_list, verbs, entities = extract_elements(doc)
        except Exception:
            nouns, chunks_list, verbs, entities = [], [], [], []
        results.append({
            'union_id': union_ids[i],
            'nouns': nouns,
            'chunks': chunks_list,
            'verbs': verbs,
            'entities': entities
        })

    cache_path = f'{CACHE_DIR}/chunk_{chunk_idx:04d}.pkl'
    with open(cache_path, 'wb') as f:
        pickle.dump(results, f)
    print(f'  Saved chunk {chunk_idx}')

print('\n✓ All chunks processed.')


Processing chunk 1/23 (films 0–1999)


chunk 0:   0%|          | 0/2000 [00:00<?, ?it/s]

  Saved chunk 0

Processing chunk 2/23 (films 2000–3999)


chunk 1:   0%|          | 0/2000 [00:00<?, ?it/s]

  Saved chunk 1

Processing chunk 3/23 (films 4000–5999)


chunk 2:   0%|          | 0/2000 [00:00<?, ?it/s]

... (37 outputs left out)

chunk 21:   0%|          | 0/2000 [00:00<?, ?it/s]

  Saved chunk 21

Processing chunk 23/23 (films 44000–45957)


chunk 22:   0%|          | 0/1958 [00:00<?, ?it/s]

  Saved chunk 22

✓ All chunks processed.


## 7. Combine Chunks and Save Parquets

Same two outputs as v1, but `elements_by_film.parquet` now has a new `chunks` column and `element_index.parquet` includes rows with `element_type='chunk'`.

In [20]:
all_results = []
cache_files = sorted([f for f in os.listdir(CACHE_DIR) if f.startswith('chunk_')])
print(f'Loading {len(cache_files)} cache files...')
for f in tqdm(cache_files):
    with open(f'{CACHE_DIR}/{f}', 'rb') as fp:
        all_results.extend(pickle.load(fp))

print(f'Total film results: {len(all_results):,}')

by_film = pd.DataFrame(all_results)
by_film['n_nouns']    = by_film['nouns'].str.len()
by_film['n_chunks']   = by_film['chunks'].str.len()
by_film['n_verbs']    = by_film['verbs'].str.len()
by_film['n_entities'] = by_film['entities'].str.len()

print(f'\nElement count statistics:')
print(by_film[['n_nouns', 'n_chunks', 'n_verbs', 'n_entities']].describe().round(1).to_string())

Loading 23 cache files...


  0%|          | 0/23 [00:00<?, ?it/s]

Total film results: 45,958

Element count statistics:
       n_nouns  n_chunks  n_verbs  n_entities
count  45958.0   45958.0  45958.0     45958.0
mean      60.4      17.8     36.0         8.9
std       53.8      15.6     34.8        11.5
min        0.0       0.0      0.0         0.0
25%       21.0       7.0     10.0         2.0
50%       41.0      13.0     23.0         5.0
75%       87.0      24.0     53.0        11.0
max      874.0     317.0    547.0       198.0


In [21]:
# Save elements_by_film.parquet (with new 'chunks' column)
by_film_path = f'{OUTPUT_DIR}/elements_by_film.parquet'
by_film.to_parquet(by_film_path, engine='pyarrow', compression='snappy')
print(f'Saved: {by_film_path}  ({os.path.getsize(by_film_path)/1e6:.1f} MB)')

# Build element_index (flattened, with new chunk rows)
print('\nBuilding flattened element index...')
rows = []
for r in tqdm(all_results):
    uid = r['union_id']
    for n in r['nouns']:
        rows.append((uid, n, 'noun'))
    for c in r['chunks']:
        rows.append((uid, c, 'chunk'))
    for v in r['verbs']:
        rows.append((uid, v, 'verb'))
    for e in r['entities']:
        rows.append((uid, e['text'], f'entity_{e["label"]}'))

element_index = pd.DataFrame(rows, columns=['union_id', 'element', 'element_type'])
print(f'Total element rows: {len(element_index):,}')
print(f'\nBreakdown by type:')
print(element_index['element_type'].value_counts())

index_path = f'{OUTPUT_DIR}/element_index.parquet'
element_index.to_parquet(index_path, engine='pyarrow', compression='snappy')
print(f'\nSaved: {index_path}  ({os.path.getsize(index_path)/1e6:.1f} MB)')

Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02_elements/elements_by_film.parquet  (21.4 MB)

Building flattened element index...


  0%|          | 0/45958 [00:00<?, ?it/s]

Total element rows: 5,658,714

Breakdown by type:
element_type
noun                  2776891
verb                  1656455
chunk                  817074
entity_ORG             177807
entity_GPE             115125
entity_NORP             50379
entity_LOC              18449
entity_WORK_OF_ART      15535
entity_PRODUCT          14036
entity_FAC              10435
entity_EVENT             6528
Name: count, dtype: int64

Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/02_elements/element_index.parquet  (30.3 MB)


## 8. Sanity Check: Top 15 Chunks per Genre

This is the diagnostic that tells us whether chunk extraction is catching what we hoped. Look for multi-word stock elements per genre:
- Horror: `haunted house`, `dark forest`, `dead body`, `evil spirit`, `young woman`
- Western: `young man`, `small town`, `local sheriff`, `frontier town`, `train robbery`
- Sci-Fi: `space station`, `alien planet`, `time machine`, `dead body`, `space ship`
- Romance: `young woman`, `young man`, `true love`, `wedding day`, `marriage proposal`
- War: `young soldier`, `enemy line`, `prisoner war`, `command officer`
- Crime: `police officer`, `crime scene`, `young woman`, `drug dealer`

In [22]:
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western'
]

# Fix the parquet round-trip array type issue on genres_canonical
union['genres_canonical'] = union['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)

# Film -> genre list lookup
film_genres = dict(zip(union['union_id'], union['genres_canonical']))

chunks_only = element_index[element_index['element_type'] == 'chunk']
print(f'Chunk elements in index: {len(chunks_only):,}')
print(f'Unique chunks: {chunks_only["element"].nunique():,}')

Chunk elements in index: 817,074
Unique chunks: 420,949


In [23]:
print('=' * 90)
print(f'{"TOP 15 NOUN-CHUNK ELEMENTS BY GENRE (raw frequency)":^90}')
print('=' * 90)

for genre in CANONICAL_GENRES:
    genre_film_ids = [uid for uid, gs in film_genres.items() if genre in list(gs)]
    genre_chunks = chunks_only[chunks_only['union_id'].isin(genre_film_ids)]
    counter = Counter(genre_chunks['element'].tolist())
    top = counter.most_common(15)

    print(f'\n{genre.upper()} (N={len(genre_film_ids):,} films, {len(genre_chunks):,} chunk instances)')
    for row_start in range(0, 15, 3):
        row_items = top[row_start:row_start+3]
        parts = [f'  {i+1+row_start:2d}. {elem:<24}({count:>5,})' for i, (elem, count) in enumerate(row_items)]
        print(''.join(parts))

                   TOP 15 NOUN-CHUNK ELEMENTS BY GENRE (raw frequency)                    

ACTION (N=7,775 films, 173,202 chunk instances)
   1. police officer          (  430)   2. united states           (  341)   3. hong kong               (  302)
   4. los angeles             (  241)   5. new york                (  238)   6. best friend             (  218)
   7. young man               (  215)   8. old man                 (  187)   9. gang member             (  182)
  10. old friend              (  180)  11. police station          (  178)  12. younger brother         (  175)
  13. new york city           (  169)  14. young woman             (  153)  15. police force            (  141)

ADVENTURE (N=6,146 films, 146,550 chunk instances)
   1. united states           (  288)   2. new york                (  234)   3. police officer          (  185)
   4. los angeles             (  185)   5. hong kong               (  184)   6. best friend             (  178)
   7. young man         

In [24]:
# Also show single-word noun top 15 for comparison (verify v1 results replicate)
nouns_only = element_index[element_index['element_type'] == 'noun']
print('=' * 90)
print(f'{"TOP 10 SINGLE-WORD NOUNS BY GENRE (frequency, should match v1)":^90}')
print('=' * 90)

for genre in CANONICAL_GENRES:
    genre_film_ids = [uid for uid, gs in film_genres.items() if genre in list(gs)]
    genre_nouns = nouns_only[nouns_only['union_id'].isin(genre_film_ids)]
    counter = Counter(genre_nouns['element'].tolist())
    top = counter.most_common(10)

    top_str = ', '.join([f'{e}({c})' for e, c in top])
    print(f'{genre:<18} {top_str}')

              TOP 10 SINGLE-WORD NOUNS BY GENRE (frequency, should match v1)              
Action             man(8507), father(4019), friend(3990), life(3551), son(3326), death(3190), time(3179), brother(3082), group(2784), money(2729)
Adventure          man(6694), friend(3262), father(2998), time(2695), life(2636), group(2425), death(2262), son(2132), ship(2062), family(1973)
Animation          bug(2041), friend(1575), cat(1466), time(1405), man(1343), house(1264), head(1017), world(986), dog(982), life(981)
Comedy             man(8992), friend(8964), father(6766), life(6553), house(6410), love(6146), time(5684), family(5164), money(5079), woman(5016)
Crime              man(5625), police(3514), money(3325), friend(2903), car(2658), house(2561), life(2535), wife(2299), murder(2273), death(2190)
Drama              man(16390), life(13525), father(12717), friend(12564), love(10439), family(10353), mother(10088), son(9326), time(9130), house(8916)
Family             father(2831), friend(2

## 9. Summary

If the chunk top-15s show multi-word stock phrases like `haunted house`, `young woman`, `small town`, `police officer`, `space station`, the extraction is working. If they're dominated by generic combinations like `old man`, `young woman`, `same time`, that's expected at the frequency level — Notebook 03 will downweight them via TF-IDF, Dunning's G², and the breadth penalty.

**Outputs ready for Notebook 03:**
- `elements_by_film.parquet` — now with `chunks` column
- `element_index.parquet` — now includes `element_type='chunk'` rows

**Next:** Re-run Notebook 03 on the new element index. The stockiness math is element-type-aware, so 'haunted house' (chunk) and 'house' (noun) get independent scores. The expected result: chunk-type elements dominate the high-stockiness lists for genres where multi-word phrases carry distinct meaning (Horror, Romance, War), while single-word nouns still dominate for genres with shorter conventional vocabulary (Western, Sci-Fi).